# EMET2007 Week 12: GDP Forecasting with AR Models

## Learning Objectives

By the end of this tutorial, you will be able to:
1. Apply unit root testing to real GDP data
2. Transform non-stationary GDP series into stationary growth rates
3. Implement pseudo out-of-sample AR(1) forecasts for GDP growth
4. Extend the forecasting framework to AR(k) models
5. Convert growth rate forecasts back to level forecasts

---

## Introduction

In this final workshop, we apply everything we've learned about time series analysis to produce a real-world economic forecast. Using US real GDP data from FRED (Federal Reserve Economic Data), we will:

1. **Test for stationarity** - Does GDP have a unit root?
2. **Transform if needed** - Convert to growth rates
3. **Build forecasting models** - AR(1) and AR(4)
4. **Generate forecasts** - Predict next quarter's GDP

---

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/juergenmeinecke/EMET2007/refs/heads/main/datasets/gdp_us_2026.csv')

### Exercise 1: Set Up the Time Index

Run the cells below to create a quarterly time index and rename the GDP column.
This is required for all period-based indexing in the rest of the notebook.

In [ ]:
df['date'] = pd.to_datetime(df['DATE'], format='%Y-%m-%d')
df.index = pd.DatetimeIndex(df.date, name='quarter').to_period('Q')

In [ ]:
df = df.rename({'GDPC1': 'gdp'}, axis=1)
df[['date', 'gdp']].head()

---

## Part 1: Initial Data Exploration

### Exercise 2: Plot the GDP Time Series

Before modeling, we should visualise the data to understand its characteristics.

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(df.date, df.gdp)
# ax.set_xlabel('Time')
# ax.set_ylabel('GDP ($bn, chained 2012 dollars)')
# ax.set_title('US Real GDP Time Series')
# plt.show()

**Question:** What do you observe about the GDP time series? Does it appear stationary?

### Exercise 3: Focus on Recent History

Let's zoom in on the period from 2000 onwards to see recent economic events more clearly:

In [ ]:
# df_2000 = df[df.index >= '2000Q1']
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(df_2000.date, df_2000.gdp)
# ax.set_xlabel('Time')
# ax.set_ylabel('GDP ($bn, chained 2012 dollars)')
# ax.set_title('US Real GDP (2000-present)')
# plt.show()

**Observations:** This plot clearly shows:
- The 2007-2009 Global Financial Crisis (GFC)
- The sharp COVID-19 recession in 2020
- The subsequent recovery

---

## Part 2: Testing for Unit Roots

### Background: Why Unit Root Testing Matters

If GDP has a unit root (is non-stationary), then:
- Standard regression results are unreliable
- The AR coefficient near 1 doesn't have its usual interpretation
- Forecasts will be biased

### Exercise 4: Naïve AR(1) Model

First, let's fit a simple AR(1) model to GDP levels and examine the coefficient:

In [ ]:
# df['l1_gdp'] = df.gdp.shift(1)
# forecast_ar1 = smf.ols('gdp ~ l1_gdp', data=df, missing='drop').fit(use_t=False)
# forecast_ar1.summary()

**Key observation:** The coefficient on `l1_gdp` is very close to 1 (approximately 1.0045). This is a warning sign of a potential unit root.

### Exercise 5: Formal Unit Root Test

We use the Augmented Dickey-Fuller (ADF) test:

- $H_0$: Unit root exists (series is non-stationary)
- $H_1$: No unit root (series is stationary)

The `regression='ct'` option includes both a constant and trend in the test regression.

In [ ]:
# adf_result = adfuller(df.gdp, maxlag=0, regression='ct')
# print(f'ADF Test Statistic: {adf_result[0]:.4f}')
# print(f'p-value:            {adf_result[1]:.4f}')
# print('Critical Values:')
# for key, val in adf_result[4].items():
#     print(f'  {key}: {val:.4f}')

**Interpretation:**
- Compare the test statistic to the 5% critical value (~-3.41)
- If test stat > critical value (less negative), we **cannot reject** the null
- The p-value also tells us the same: if p > 0.05, cannot reject $H_0$

**Conclusion:** GDP has a unit root. We cannot use the levels for forecasting.

---

## Part 3: Transforming to GDP Growth

### Exercise 6: Create the GDP Growth Rate

Since GDP has a unit root, we transform it to growth rates:

$$\text{GDP Growth}_t = 400 \times (\ln(\text{GDP}_t) - \ln(\text{GDP}_{t-1}))$$

The multiplication by 400 annualises the quarterly growth rate (4 quarters × 100 for percentage).

In [ ]:
# df['log_gdp'] = np.log(df.gdp)
# df['gdp_growth'] = 400 * (df.log_gdp - df.log_gdp.shift(1))

### Exercise 7: Plot GDP Growth

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(df.date, df.gdp_growth)
# ax.set_xlabel('Time')
# ax.set_ylabel('GDP growth rate (annualised, %)')
# ax.set_title('US Real GDP Growth Rate')
# plt.show()

In [ ]:
# Focus on 2000 onwards
# df_2000 = df[df.index >= '2000Q1']
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(df_2000.date, df_2000.gdp_growth)
# ax.set_xlabel('Time')
# ax.set_ylabel('GDP growth rate (annualised, %)')
# ax.set_title('US Real GDP Growth Rate (2000-present)')
# plt.show()

**Note:** The COVID-19 period shows extreme values (~-30% in 2020Q2, then +30% recovery). These are **annualised** rates; actual quarterly changes were about 1/4 of these values.

### Exercise 8: Pre-Pandemic View

To see historical recessions more clearly, let's exclude the pandemic period:

In [ ]:
# df_2019 = df[df.index <= '2019Q4']
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(df_2019.date, df_2019.gdp_growth)
# ax.set_xlabel('Time')
# ax.set_ylabel('GDP growth rate (annualised, %)')
# ax.set_title('US Real GDP Growth Rate (pre-pandemic)')
# ax.grid(True)
# plt.show()

**Historical recessions visible in the data:**
- 1973-75: Oil crisis
- 1980/1981-82: Energy crisis and Volcker monetary tightening
- 2001: Dotcom bust (relatively mild)
- 2007-09: Global Financial Crisis

### Exercise 9: Test GDP Growth for Unit Root

In [ ]:
# adf_growth = adfuller(df.gdp_growth.dropna(), maxlag=0, regression='ct')
# print(f'ADF Test Statistic: {adf_growth[0]:.4f}')
# print(f'p-value:            {adf_growth[1]:.6f}')
# print('Critical Values:')
# for key, val in adf_growth[4].items():
#     print(f'  {key}: {val:.4f}')

**Interpretation:**
- ADF test statistic is very negative
- It is much smaller (more negative) than the critical value of -3.41
- p-value is essentially zero

**Conclusion:** We strongly **reject** the null hypothesis. GDP growth is stationary and can be used for forecasting.

---

## Part 4: Pseudo Out-of-Sample Forecasting

### The AR(1) Forecasting Function

We use the `ar1_pseudo` function from Week 11. This function:
1. Starts at a given date
2. Estimates AR(1) using only data available at that time
3. Makes a one-step-ahead forecast
4. Moves forward one period and repeats

This simulates **real-time** forecasting performance.

In [ ]:
def ar1_pseudo(input_df, y, startdate):
    """
    Generate pseudo out-of-sample forecasts from AR(1) model.
    
    Parameters:
    - input_df: DataFrame with quarterly period index
    - y: name of variable to forecast (string)
    - startdate: first quarter to forecast (e.g., '2005Q1')
    
    Returns:
    - DataFrame with columns: forecast, forecast_error, date
    """
    df = pd.DataFrame()
    df['y'] = input_df[y]
    df['lag_y'] = df.y.shift(1)
    df['trend'] = range(len(input_df[y]))
    df['next_forecast'] = pd.Series(dtype='float64')
    
    start_period = pd.Period(startdate, freq='Q')
    
    for i, row in df.iterrows():
        if row.name < start_period - 1:
            continue
        df_sub = df[df.index <= row.name]
        ar_tmp = smf.ols('y ~ lag_y + trend', data=df_sub, missing='drop').fit()
        df_aux = {'lag_y': row.y, 'trend': row.trend}
        prediction = ar_tmp.predict(df_aux)
        df.at[i, 'next_forecast'] = prediction[0]
    
    df.loc[df.iloc[-1].name + 1, :] = np.nan
    df['forecast'] = df.next_forecast.shift(1)
    df['forecast_error'] = df.y - df.forecast
    
    results_df = df[['forecast', 'forecast_error']].copy()
    results_df['date'] = results_df.index.to_timestamp()
    return results_df

### Exercise 10: Generate AR(1) Forecasts

In [ ]:
# results_df = ar1_pseudo(df, 'gdp_growth', '2005Q1')
# results_df.tail(10)

**Understanding the output:**
- Values up to 2026:Q1 are **pseudo out-of-sample predictions** (we know the actual values)
- The value for 2026:Q2 is our **true forecast** (actual value not yet known)
- `forecast_error = actual - forecast` (NaN for the true forecast)

### Exercise 11: Plot Forecasts vs Actuals

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.plot(results_df.date, results_df.forecast, 'b-', label='AR(1) Forecast')
# df_2000 = df[df.index >= '2000Q1']
# ax.plot(df_2000.date, df_2000.gdp_growth, 'g-', label='Actual')
# ax.set_xlabel('Date')
# ax.set_ylabel('GDP growth (annualised, %)')
# ax.set_title('AR(1) Forecasts vs Actual GDP Growth')
# ax.legend()
# plt.show()

### Exercise 12: Pre-Pandemic Comparison

The pandemic makes it hard to see the forecast behaviour. Let's look at pre-2020:

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 6))
# results_df_19 = results_df[(results_df.index >= '2000Q1') & (results_df.index <= '2019Q4')]
# ax.plot(results_df_19.date, results_df_19.forecast, 'b-', label='AR(1) Forecast')
# df_19 = df[(df.index >= '2000Q1') & (df.index <= '2019Q4')]
# ax.plot(df_19.date, df_19.gdp_growth, 'g-', label='Actual')
# ax.set_xlabel('Date')
# ax.set_ylabel('GDP growth (annualised, %)')
# ax.set_title('AR(1) Forecasts vs Actual GDP Growth (2000-2019)')
# ax.legend()
# ax.grid(True)
# plt.show()

---

## Part 5: Extending to AR(k) Models

### The AR(k) Forecasting Function

We can generalise the AR(1) function to include $k$ lags:

$$Y_t = \beta_0 + \beta_1 t + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \cdots + \phi_k Y_{t-k} + u_t$$

In [ ]:
def ar_k_pseudo(input_df, y, startdate, lags=1):
    """
    Generate pseudo out-of-sample forecasts from AR(k) model.
    
    Parameters:
    - input_df: DataFrame with quarterly period index
    - y: name of variable to forecast (string)
    - startdate: first quarter to forecast (e.g., '2005Q1')
    - lags: number of lags k (default=1)
    
    Returns:
    - DataFrame with columns: forecast, forecast_error, date
    """
    df = pd.DataFrame()
    df['y'] = input_df[y]
    for k in range(1, lags + 1):
        df[f'lag{k}_y'] = df.y.shift(k)
    df['trend'] = range(len(input_df[y]))
    df['next_forecast'] = pd.Series(dtype='float64')
    
    start_period = pd.Period(startdate, freq='Q')
    
    for i, row in df.iterrows():
        if row.name < start_period - 1:
            continue
        df_sub = df[df.index <= row.name]
        
        # Build formula dynamically
        formula = 'y ~ '
        for k in range(1, lags + 1):
            formula += f'lag{k}_y + '
        formula += 'trend'
        
        ar_tmp = smf.ols(formula, data=df_sub, missing='drop').fit()
        
        # Build prediction dictionary
        df_aux = {'lag1_y': row.y, 'trend': row.trend}
        for k in range(2, lags + 1):
            df_aux[f'lag{k}_y'] = row[f'lag{k - 1}_y']
        prediction = ar_tmp.predict(df_aux)
        df.at[i, 'next_forecast'] = prediction[0]
    
    df.loc[df.iloc[-1].name + 1, :] = np.nan
    df['forecast'] = df.next_forecast.shift(1)
    df['forecast_error'] = df.y - df.forecast
    
    results_df = df[['forecast', 'forecast_error']].copy()
    results_df['date'] = results_df.index.to_timestamp()
    return results_df

### Exercise 13: Generate AR(4) Forecasts

Using 4 lags captures potential quarterly seasonal patterns in GDP growth:

In [ ]:
# results2_df = ar_k_pseudo(df, 'gdp_growth', '2005Q1', lags=4)
# results2_df.tail(10)

### Exercise 14: Compare AR(1) and AR(4) Forecasts

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.plot(results2_df.date, results2_df.forecast, 'b-', label='AR(4) Forecast')
# df_2000 = df[df.index >= '2000Q1']
# ax.plot(df_2000.date, df_2000.gdp_growth, 'g-', label='Actual')
# ax.set_xlabel('Date')
# ax.set_ylabel('GDP growth (annualised, %)')
# ax.set_title('AR(4) Forecasts vs Actual GDP Growth')
# ax.legend()
# plt.show()

---

## Part 6: Converting Back to GDP Levels

### Exercise 15: Extract Growth Rate Forecasts

Our forecasts are for GDP **growth**. To get a GDP **level** forecast:

In [ ]:
# Extract the 2026:Q2 forecast (last row with a forecast)
# ar1_forecast_gr = results_df.forecast['2026Q2']
# ar4_forecast_gr = results2_df.forecast['2026Q2']
# print(f'AR(1) growth forecast: {ar1_forecast_gr:.2f}%')
# print(f'AR(4) growth forecast: {ar4_forecast_gr:.2f}%')

### Exercise 16: Convert to GDP Level

To convert growth back to level, we invert the transformation:

$$\text{GDP}_{t+1} = \exp\left(\ln(\text{GDP}_t) + \frac{\text{growth forecast}}{400}\right)$$

In [ ]:
# ar1_forecast = np.exp(np.log(df.gdp['2026Q1']) + (ar1_forecast_gr / 400))
# ar4_forecast = np.exp(np.log(df.gdp['2026Q1']) + (ar4_forecast_gr / 400))
# print(f'AR(1) GDP forecast for 2026:Q2: ${ar1_forecast:,.0f} billion')
# print(f'AR(4) GDP forecast for 2026:Q2: ${ar4_forecast:,.0f} billion')

---

## Summary

### Key Findings

**Unit Root Tests:**
- GDP in levels: Has a unit root (non-stationary) → cannot reject $H_0$
- GDP growth: No unit root (stationary) → strongly reject $H_0$

**Forecasting Results:**
- AR(1) and AR(4) models produce similar forecasts
- Both predict continued positive but moderate growth

### Workflow Recap

1. **Test for stationarity** using ADF test
2. **Transform if needed** (first differences / growth rates)
3. **Re-test** to confirm stationarity
4. **Build AR model** and generate forecasts
5. **Convert back** to original units if needed

### Limitations

- AR models are **univariate** – they only use past values of the series itself
- They cannot incorporate other economic indicators (interest rates, employment, etc.)
- Forecast uncertainty increases with horizon
- Extreme events (like COVID-19) are impossible to predict

### Extensions

More sophisticated approaches (covered in advanced courses) include:
- **ARMA/ARIMA models** – combine AR with Moving Average terms
- **VAR models** – multivariate systems
- **GARCH models** – model time-varying volatility
- **Machine learning** – neural networks, random forests for forecasting